In [7]:
import os
from pathlib import Path

import nibabel as nib
import torch
from nibabel.filebasedimages import ImageFileError
from totalsegmentator.config import has_valid_license_offline, set_license_number
from totalsegmentator.python_api import totalsegmentator

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != "notebooks":
    NOTEBOOK_DIR = NOTEBOOK_DIR / "notebooks"

dataset_dir = NOTEBOOK_DIR / "nifti_data"
TASK = "heartchambers_highres"
START_FROM_TAVI = "TAVI_318"
LICENSE_NUMBER = "aca_BMG3E3I7YN8V5Z"

if not dataset_dir.exists():
    raise FileNotFoundError(f"Dataset directory not found: {dataset_dir}")

if LICENSE_NUMBER:
    set_license_number(LICENSE_NUMBER, skip_validation=True)

license_status, license_message = has_valid_license_offline()
if license_status != "yes":
    raise RuntimeError(
        "The TotalSegmentator task 'heartchambers_highres' requires a TotalSegmentator license "
        "to create heart_myocardium.nii.gz. Get the free academic license from "
        "https://backend.totalsegmentator.com/license-academic/ and either set the "
        "TOTALSEG_LICENSE_NUMBER environment variable before starting Jupyter or run in a notebook cell: "
        "import os; os.environ['TOTALSEG_LICENSE_NUMBER'] = '<your-license-number>'. "
        f"Current license status: {license_status}; {license_message}"
    )

if not torch.backends.mps.is_available():
    raise RuntimeError(
        "MPS is not available in this Python environment. "
        "Run this notebook on Apple Silicon with an MPS-enabled PyTorch build."
    )

patient_dirs = sorted(p for p in dataset_dir.iterdir() if p.is_dir() and p.name.startswith("TAVI_"))
if START_FROM_TAVI:
    patient_dirs = [p for p in patient_dirs if p.name >= START_FROM_TAVI]
    print(f"Starting from {START_FROM_TAVI}: {len(patient_dirs)} patient folders queued")
processed = []
skipped = []
missing_ct = []
unreadable_ct = []

for patient_dir in patient_dirs:
    ct_file = patient_dir / "CT_LATE.nii.gz"
    output_dir = patient_dir / "TotalSegmentator" / "CT_LATE" / TASK
    myocardium_mask = output_dir / "heart_myocardium.nii.gz"

    if not ct_file.exists():
        missing_ct.append(patient_dir.name)
        continue

    try:
        nib.load(str(ct_file))
    except ImageFileError as exc:
        unreadable_ct.append((patient_dir.name, str(exc)))
        print(f"Skipping {patient_dir.name}: CT_LATE unreadable ({exc})")
        continue

    if myocardium_mask.exists():
        skipped.append(patient_dir.name)
        continue

    output_dir.mkdir(parents=True, exist_ok=True)
    print(f"Processing {patient_dir.name} -> {myocardium_mask}")

    try:
        totalsegmentator(
            input=str(ct_file),
            output=str(output_dir),
            task="heartchambers_highres",
            device="mps",
            quiet=False,
        )
    except SystemExit as exc:
        raise RuntimeError(
            f"TotalSegmentator failed for {patient_dir.name}. "
            "Check that the heartchambers_highres license is configured and valid, "
            "then rerun this cell."
        ) from exc
    processed.append(patient_dir.name)

print(f"Processed: {len(processed)}")
print(f"Skipped existing masks: {len(skipped)}")
print(f"Missing CT_LATE files: {len(missing_ct)}")
print(f"Unreadable CT_LATE files: {len(unreadable_ct)}")
for patient_id, reason in unreadable_ct:
    print(f"  {patient_id}: {reason}")

Starting from TAVI_318: 34 patient folders queued
Skipping TAVI_318: CT_LATE unreadable (File /Users/ricca/Desktop/Health_Informatics_Internship_2026/notebooks/nifti_data/TAVI_318/CT_LATE.nii.gz is not a gzip file)
Skipping TAVI_320: CT_LATE unreadable (File /Users/ricca/Desktop/Health_Informatics_Internship_2026/notebooks/nifti_data/TAVI_320/CT_LATE.nii.gz is not a gzip file)
Processing TAVI_321 -> /Users/ricca/Desktop/Health_Informatics_Internship_2026/notebooks/nifti_data/TAVI_321/TotalSegmentator/CT_LATE/heartchambers_highres/heart_myocardium.nii.gz

If you use this tool please cite: https://pubs.rsna.org/doi/10.1148/ryai.230024

Generating rough segmentation for cropping...
  (Using more robust (but slower) 3mm model for cropping.)
Resampling...
  Resampled in 0.58s
Predicting...


100%|██████████| 1/1 [00:00<00:00,  1.13it/s]


  Predicted in 4.53s
Resampling...
  cropping from (512, 512, 57) to (448, 488, 44)
Predicting...


100%|██████████| 12/12 [00:10<00:00,  1.20it/s]


  Predicted in 18.15s
Applying postprocessing: remove outside of crop mask...
  Applied in 0.11s
Saving segmentations...
Creating heart_ventricle_left.nii.gzCreating heart_atrium_left.nii.gz

Creating heart_myocardium.nii.gzCreating heart_ventricle_right.nii.gzCreating heart_atrium_right.nii.gz


Creating aorta.nii.gz
Creating pulmonary_artery.nii.gz
  Saved in 4.02s
Processing TAVI_323 -> /Users/ricca/Desktop/Health_Informatics_Internship_2026/notebooks/nifti_data/TAVI_323/TotalSegmentator/CT_LATE/heartchambers_highres/heart_myocardium.nii.gz

If you use this tool please cite: https://pubs.rsna.org/doi/10.1148/ryai.230024

Generating rough segmentation for cropping...
  (Using more robust (but slower) 3mm model for cropping.)
Resampling...
  Resampled in 0.48s
Predicting...


...

  Predicted in 20.31s
Applying postprocessing: remove outside of crop mask...
  Applied in 0.14s
Saving segmentations...
Creating heart_myocardium.nii.gz
Creating aorta.nii.gzCreating heart_atrium_right.nii.gz
Creating heart_ventricle_left.nii.gz

Creating heart_ventricle_right.nii.gz
Creating heart_atrium_left.nii.gz
Creating pulmonary_artery.nii.gz
  Saved in 3.68s
Processing TAVI_377 -> /Users/ricca/Desktop/Health_Informatics_Internship_2026/notebooks/nifti_data/TAVI_377/TotalSegmentator/CT_LATE/heartchambers_highres/heart_myocardium.nii.gz

If you use this tool please cite: https://pubs.rsna.org/doi/10.1148/ryai.230024

Generating rough segmentation for cropping...
  (Using more robust (but slower) 3mm model for cropping.)
Resampling...
  Resampled in 0.48s
Predicting...


100%|██████████| 1/1 [00:00<00:00,  1.47it/s]


  Predicted in 4.04s
Resampling...
  cropping from (512, 512, 46) to (495, 416, 43)
Predicting...


100%|██████████| 12/12 [00:09<00:00,  1.21it/s]


  Predicted in 17.22s
Applying postprocessing: remove outside of crop mask...
  Applied in 0.09s
Saving segmentations...
Creating heart_myocardium.nii.gz
Creating aorta.nii.gz
Creating heart_ventricle_right.nii.gz
Creating heart_ventricle_left.nii.gz
Creating heart_atrium_left.nii.gz
Creating heart_atrium_right.nii.gz
Creating pulmonary_artery.nii.gz
  Saved in 3.08s
Processed: 31
Skipped existing masks: 0
Missing CT_LATE files: 0
Unreadable CT_LATE files: 3
  TAVI_318: File /Users/ricca/Desktop/Health_Informatics_Internship_2026/notebooks/nifti_data/TAVI_318/CT_LATE.nii.gz is not a gzip file
  TAVI_320: File /Users/ricca/Desktop/Health_Informatics_Internship_2026/notebooks/nifti_data/TAVI_320/CT_LATE.nii.gz is not a gzip file
  TAVI_332: File /Users/ricca/Desktop/Health_Informatics_Internship_2026/notebooks/nifti_data/TAVI_332/CT_LATE.nii.gz is not a gzip file
